In [1]:
import pandas as pd
import numpy as np

# Load the results data
results = pd.read_csv('data/raw/results.csv')
results['date'] = pd.to_datetime(results['date'])
results = results.sort_values('date').reset_index(drop=True)

# Drop rows with missing scores
results = results.dropna(subset=['home_score', 'away_score'])

print("Shape:", results.shape)
print("Date range:", results['date'].min(), "to", results['date'].max())
print("Done!")

Shape: (49409, 9)
Date range: 1872-11-30 00:00:00 to 2026-06-12 00:00:00
Done!


In [2]:
# Elo Rating System
# Starting Elo for all teams, K-factor controls how fast ratings change

ELO_START = 1000
K = 30

elo_ratings = {}

def get_elo(team):
    return elo_ratings.get(team, ELO_START)

def update_elo(winner, loser, draw=False):
    elo_w = get_elo(winner)
    elo_l = get_elo(loser)
    
    expected_w = 1 / (1 + 10 ** ((elo_l - elo_w) / 400))
    expected_l = 1 - expected_w
    
    actual_w = 0.5 if draw else 1
    actual_l = 0.5 if draw else 0
    
    elo_ratings[winner] = elo_w + K * (actual_w - expected_w)
    elo_ratings[loser] = elo_l + K * (actual_l - expected_l)

# Process every match chronologically and record Elo BEFORE the match
home_elos, away_elos = [], []

for _, row in results.iterrows():
    home = row['home_team']
    away = row['away_team']
    
    home_elos.append(get_elo(home))
    away_elos.append(get_elo(away))
    
    if row['home_score'] > row['away_score']:
        update_elo(home, away)
    elif row['home_score'] < row['away_score']:
        update_elo(away, home)
    else:
        update_elo(home, away, draw=True)

results['home_elo'] = home_elos
results['away_elo'] = away_elos
results['elo_diff'] = results['home_elo'] - results['away_elo']

print("Top 10 Elo ratings right now:")
print(sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)[:10])

Top 10 Elo ratings right now:
[('Argentina', 1547.0278155109747), ('Spain', 1545.3370517540634), ('France', 1487.8299716695165), ('Portugal', 1454.1799161938873), ('Brazil', 1448.2316560357642), ('England', 1444.098723722469), ('Germany', 1443.4841745514022), ('Colombia', 1430.3354002921526), ('Netherlands', 1417.3931145783467), ('Japan', 1411.349745725385)]


In [3]:
# Recent form - win rate over last 10 matches for each team
def get_recent_form(results, n=10):
    form = {}
    home_form, away_form = [], []
    
    for _, row in results.iterrows():
        home = row['home_team']
        away = row['away_team']
        
        # Get last n results for each team
        home_history = form.get(home, [])
        away_history = form.get(away, [])
        
        home_form.append(np.mean(home_history[-n:]) if home_history else 0.5)
        away_form.append(np.mean(away_history[-n:]) if away_history else 0.5)
        
        # Update form: 1 = win, 0.5 = draw, 0 = loss
        if row['home_score'] > row['away_score']:
            form.setdefault(home, []).append(1)
            form.setdefault(away, []).append(0)
        elif row['home_score'] < row['away_score']:
            form.setdefault(home, []).append(0)
            form.setdefault(away, []).append(1)
        else:
            form.setdefault(home, []).append(0.5)
            form.setdefault(away, []).append(0.5)
    
    return home_form, away_form

results['home_form'], results['away_form'] = get_recent_form(results)
results['form_diff'] = results['home_form'] - results['away_form']

print("Sample features:")
print(results[['date','home_team','away_team','home_elo','away_elo','elo_diff','home_form','away_form']].tail(10))

Sample features:
            date      home_team                 away_team     home_elo  \
49399 2026-06-09         Angola  Central African Republic  1049.027020   
49400 2026-06-09        Hungary                Kazakhstan  1255.855177   
49401 2026-06-10        Bolivia                   Algeria  1129.324663   
49402 2026-06-10        England                Costa Rica  1438.964782   
49403 2026-06-10       Portugal                   Nigeria  1446.466529   
49404 2026-06-10    Afghanistan                  Pakistan   734.373330   
49405 2026-06-11    South Korea            Czech Republic  1321.970269   
49406 2026-06-11         Mexico              South Africa  1375.040393   
49407 2026-06-12         Canada    Bosnia and Herzegovina  1313.685669   
49408 2026-06-12  United States                  Paraguay  1310.927219   

          away_elo    elo_diff  home_form  away_form  
49399   877.409506  171.617514       0.45       0.30  
49400   988.883212  266.971965       0.65       0.45  
494